### precipitation

In [3]:
# ===============================================================
# NEX-GDDP-CMIP6  |  Monthly cumulative precipitation (mm/month)
# AOI: Tekeze Basin
# Historical : 1981-2014
# Future SSP245 & SSP585 : 2030-2070
# Export format : GeoTIFF to Google Drive
# ===============================================================

import ee

# ===============================================================
# INITIALIZE EARTH ENGINE
# ===============================================================

ee.Authenticate()      # Run once interactively
ee.Initialize(project='gee-research123456')

# ===============================================================
# STUDY AREA
# ===============================================================

AOI = ee.FeatureCollection(
    "projects/gee-research123456/assets/tekeze"
)

# ===============================================================
# EXPORT SETTINGS
# ===============================================================

SCALE = 25000

# Main Google Drive folder
MAIN_FOLDER = 'NEXGDDP_CMIP6_Tekeze'

# ===============================================================
# MODELS
# ===============================================================

MODELS = [
    
    'GFDL-CM4',
    'NorESM2-MM',
    
]

# ===============================================================
# SCENARIOS
# ===============================================================

SCENARIOS = [
    'historical',
    'ssp245',
    'ssp585'
]

# ===============================================================
# PERIODS
# ===============================================================

PERIODS = {

    'historical': {
        'start': 1981,
        'end': 2014
    },

    'future': {
        'start': 2030,
        'end': 2050
    }
}

# ===============================================================
# HELPER FUNCTIONS
# ===============================================================

def convert_to_mm_per_day_and_clip(img):

    """
    Convert precipitation:
    kg m-2 s-1  -->  mm/day
    """

    return (
        img.select('pr')
        .multiply(86400)
        .clip(AOI)
        .copyProperties(img, img.propertyNames())
    )

# ---------------------------------------------------------------

def monthly_sum_for_year(model, scenario, year):

    """
    Create monthly cumulative precipitation
    for one year.
    """

    start = ee.Date.fromYMD(year, 1, 1)
    end = start.advance(1, 'year')

    daily = (
        ee.ImageCollection('NASA/GDDP-CMIP6')
        .filter(ee.Filter.eq('model', model))
        .filter(ee.Filter.eq('scenario', scenario))
        .filterDate(start, end)
        .map(convert_to_mm_per_day_and_clip)
    )

    months = ee.List.sequence(1, 12)

    def make_month(m):

        m = ee.Number(m)

        m_start = ee.Date.fromYMD(year, m, 1)
        m_end = m_start.advance(1, 'month')

        monthly = (
            daily
            .filterDate(m_start, m_end)
            .sum()
            .set({
                'year': year,
                'month': m,
                'scenario': scenario,
                'model': model,
                'system:time_start': m_start.millis()
            })
        )

        return monthly

    return ee.ImageCollection(months.map(make_month))

# ---------------------------------------------------------------

def export_monthly_collection(
        coll,
        model,
        scenario,
        folder,
        region,
        scale):

    """
    Export monthly images to Google Drive.

    Folder structure:

    NEXGDDP_CMIP6_Tekeze/
        ACCESS-CM2/
            historical/
            ssp245/
            ssp585/

        CESM2/
            historical/
            ssp245/
            ssp585/
    """

    size = coll.size().getInfo()
    lst = coll.toList(size)

    # -----------------------------------------------------------
    # Google Drive subfolder
    # -----------------------------------------------------------

    drive_folder = f'{folder}/{model}/{scenario}'

    for i in range(size):

        img = ee.Image(lst.get(i))

        year = img.get('year').getInfo()
        month = img.get('month').getInfo()

        mm = f'{int(month):02d}'

        # Final filename
        filename = f'{year}_{mm}'

        task = ee.batch.Export.image.toDrive(

            image=img,

            description=f'{model}_{scenario}_{year}_{mm}',

            folder=drive_folder,

            fileNamePrefix=filename,

            region=region,

            scale=scale,

            maxPixels=1e13,

            fileFormat='GeoTIFF'
        )

        task.start()

        print(f'✅ Started export: {drive_folder}/{filename}.tif')

# ===============================================================
# MAIN PROCESS
# ===============================================================

for model in MODELS:

    print('====================================================')
    print(f'Processing MODEL: {model}')
    print('====================================================')

    for scenario in SCENARIOS:

        # =======================================================
        # HISTORICAL
        # =======================================================

        if scenario == 'historical':

            period = PERIODS['historical']

            print(
                f'Processing {model} | historical | '
                f'{period["start"]}-{period["end"]}'
            )

            for year in range(
                    period['start'],
                    period['end'] + 1):

                monthly = monthly_sum_for_year(
                    model,
                    scenario,
                    year
                )

                export_monthly_collection(
                    monthly,
                    model,
                    scenario,
                    MAIN_FOLDER,
                    AOI.geometry(),
                    SCALE
                )

        # =======================================================
        # FUTURE SSP245 + SSP585
        # =======================================================

        else:

            period = PERIODS['future']

            print(
                f'Processing {model} | {scenario} | '
                f'{period["start"]}-{period["end"]}'
            )

            for year in range(
                    period['start'],
                    period['end'] + 1):

                monthly = monthly_sum_for_year(
                    model,
                    scenario,
                    year
                )

                export_monthly_collection(
                    monthly,
                    model,
                    scenario,
                    MAIN_FOLDER,
                    AOI.geometry(),
                    SCALE
                )

# ===============================================================
# FINISHED
# ===============================================================

print('====================================================')
print('✅ ALL EXPORT TASKS SUBMITTED')
print('Check Tasks tab in Google Earth Engine')
print('https://code.earthengine.google.com/tasks')
print('====================================================')

Processing MODEL: GFDL-CM4
Processing GFDL-CM4 | historical | 1981-2014
✅ Started export: NEXGDDP_CMIP6_Tekeze/GFDL-CM4/historical/1981_01.tif
✅ Started export: NEXGDDP_CMIP6_Tekeze/GFDL-CM4/historical/1981_02.tif
✅ Started export: NEXGDDP_CMIP6_Tekeze/GFDL-CM4/historical/1981_03.tif
✅ Started export: NEXGDDP_CMIP6_Tekeze/GFDL-CM4/historical/1981_04.tif
✅ Started export: NEXGDDP_CMIP6_Tekeze/GFDL-CM4/historical/1981_05.tif
✅ Started export: NEXGDDP_CMIP6_Tekeze/GFDL-CM4/historical/1981_06.tif
✅ Started export: NEXGDDP_CMIP6_Tekeze/GFDL-CM4/historical/1981_07.tif
✅ Started export: NEXGDDP_CMIP6_Tekeze/GFDL-CM4/historical/1981_08.tif
✅ Started export: NEXGDDP_CMIP6_Tekeze/GFDL-CM4/historical/1981_09.tif
✅ Started export: NEXGDDP_CMIP6_Tekeze/GFDL-CM4/historical/1981_10.tif
✅ Started export: NEXGDDP_CMIP6_Tekeze/GFDL-CM4/historical/1981_11.tif
✅ Started export: NEXGDDP_CMIP6_Tekeze/GFDL-CM4/historical/1981_12.tif
✅ Started export: NEXGDDP_CMIP6_Tekeze/GFDL-CM4/historical/1982_01.tif
✅ Sta

### temperature

In [2]:
# ===============================================================
# NEX-GDDP-CMIP6  |  Monthly mean temperature (degrees C/month)
# AOI: Tekeze Basin
# Historical : 1981-2014
# Future SSP245 & SSP585 : 2030-2050
# Export format : GeoTIFF to Google Drive
# ===============================================================

import ee

# ===============================================================
# INITIALIZE EARTH ENGINE
# ===============================================================

ee.Authenticate()      # Run once interactively
ee.Initialize(project='gee-research123456')

# ===============================================================
# STUDY AREA
# ===============================================================

AOI = ee.FeatureCollection(
    "projects/gee-research123456/assets/tekeze"
)

# ===============================================================
# EXPORT SETTINGS
# ===============================================================

SCALE = 25000

# Main Google Drive folder
MAIN_FOLDER = 'NEXGDDP_CMIP6_Tekeze'

# ===============================================================
# MODELS
# ===============================================================

MODELS = [
    
    'INM-CM5-0',
]
#'GFDL-ESM4',
#'GFDL-CM4',
#'NorESM2-MM',
#'EC-Earth3',
# ===============================================================
# SCENARIOS
# ===============================================================

SCENARIOS = [
    'historical',
    'ssp245',
    'ssp585'
]

# ===============================================================
# PERIODS
# ===============================================================

PERIODS = {

    'historical': {
        'start': 1981,
        'end': 2014
    },

    'future': {
        'start': 2030,
        'end': 2050
    }
}

# ===============================================================
# HELPER FUNCTIONS
# ===============================================================

def convert_tas_to_celsius_and_clip(img):

    """
    Convert temperature:
    Kelvin  -->  degrees Celsius  (subtract 273.15)
    """

    return (
        img.select('tas')
        .subtract(273.15)
        .clip(AOI)
        .copyProperties(img, img.propertyNames())
    )

# ---------------------------------------------------------------

def monthly_mean_for_year(model, scenario, year):

    """
    Create monthly mean temperature (degrees C)
    for one year.
    """

    start = ee.Date.fromYMD(year, 1, 1)
    end   = start.advance(1, 'year')

    daily = (
        ee.ImageCollection('NASA/GDDP-CMIP6')
        .filter(ee.Filter.eq('model', model))
        .filter(ee.Filter.eq('scenario', scenario))
        .filterDate(start, end)
        .map(convert_tas_to_celsius_and_clip)
    )

    months = ee.List.sequence(1, 12)

    def make_month(m):

        m = ee.Number(m)

        m_start = ee.Date.fromYMD(year, m, 1)
        m_end   = m_start.advance(1, 'month')

        monthly = (
            daily
            .filterDate(m_start, m_end)
            .mean()                          # mean for temperature
            .set({
                'year':             year,
                'month':            m,
                'scenario':         scenario,
                'model':            model,
                'system:time_start': m_start.millis()
            })
        )

        return monthly

    return ee.ImageCollection(months.map(make_month))

# ---------------------------------------------------------------

def export_monthly_collection(
        coll,
        model,
        scenario,
        folder,
        region,
        scale):

    """
    Export monthly images to Google Drive.

    Folder structure:

    NEXGDDP_CMIP6_Tekeze/
        GFDL-ESM4/
            historical/
                tas/
            ssp245/
                tas/
            ssp585/
                tas/

        GFDL-CM4/
            historical/
                tas/
            ...
    """

    size = coll.size().getInfo()
    lst  = coll.toList(size)

    # -----------------------------------------------------------
    # Google Drive subfolder — tas in separate subfolder
    # -----------------------------------------------------------

    drive_folder = f'{folder}/{model}/{scenario}/tas'

    for i in range(size):

        img = ee.Image(lst.get(i))

        year  = img.get('year').getInfo()
        month = img.get('month').getInfo()

        mm = f'{int(month):02d}'

        # Final filename
        filename = f'{year}_{mm}'

        task = ee.batch.Export.image.toDrive(

            image          = img,

            description    = f'{model}_{scenario}_tas_{year}_{mm}',

            folder         = drive_folder,

            fileNamePrefix = filename,

            region         = region,

            scale          = scale,

            maxPixels      = 1e13,

            fileFormat     = 'GeoTIFF'
        )

        task.start()

        print(f'✅ Started export: {drive_folder}/{filename}.tif')

# ===============================================================
# MAIN PROCESS
# ===============================================================

for model in MODELS:

    print('====================================================')
    print(f'Processing MODEL: {model}')
    print('====================================================')

    for scenario in SCENARIOS:

        # =======================================================
        # HISTORICAL
        # =======================================================

        if scenario == 'historical':

            period = PERIODS['historical']

            print(
                f'Processing {model} | historical | '
                f'{period["start"]}-{period["end"]}'
            )

            for year in range(
                    period['start'],
                    period['end'] + 1):

                monthly = monthly_mean_for_year(
                    model,
                    scenario,
                    year
                )

                export_monthly_collection(
                    monthly,
                    model,
                    scenario,
                    MAIN_FOLDER,
                    AOI.geometry(),
                    SCALE
                )

        # =======================================================
        # FUTURE SSP245 + SSP585
        # =======================================================

        else:

            period = PERIODS['future']

            print(
                f'Processing {model} | {scenario} | '
                f'{period["start"]}-{period["end"]}'
            )

            for year in range(
                    period['start'],
                    period['end'] + 1):

                monthly = monthly_mean_for_year(
                    model,
                    scenario,
                    year
                )

                export_monthly_collection(
                    monthly,
                    model,
                    scenario,
                    MAIN_FOLDER,
                    AOI.geometry(),
                    SCALE
                )

# ===============================================================
# FINISHED
# ===============================================================

print('====================================================')
print('✅ ALL EXPORT TASKS SUBMITTED')
print('Check Tasks tab in Google Earth Engine')
print('https://code.earthengine.google.com/tasks')
print('====================================================')

Processing MODEL: INM-CM5-0
Processing INM-CM5-0 | historical | 1981-2014
✅ Started export: NEXGDDP_CMIP6_Tekeze/INM-CM5-0/historical/tas/1981_01.tif
✅ Started export: NEXGDDP_CMIP6_Tekeze/INM-CM5-0/historical/tas/1981_02.tif
✅ Started export: NEXGDDP_CMIP6_Tekeze/INM-CM5-0/historical/tas/1981_03.tif
✅ Started export: NEXGDDP_CMIP6_Tekeze/INM-CM5-0/historical/tas/1981_04.tif
✅ Started export: NEXGDDP_CMIP6_Tekeze/INM-CM5-0/historical/tas/1981_05.tif
✅ Started export: NEXGDDP_CMIP6_Tekeze/INM-CM5-0/historical/tas/1981_06.tif
✅ Started export: NEXGDDP_CMIP6_Tekeze/INM-CM5-0/historical/tas/1981_07.tif
✅ Started export: NEXGDDP_CMIP6_Tekeze/INM-CM5-0/historical/tas/1981_08.tif
✅ Started export: NEXGDDP_CMIP6_Tekeze/INM-CM5-0/historical/tas/1981_09.tif
✅ Started export: NEXGDDP_CMIP6_Tekeze/INM-CM5-0/historical/tas/1981_10.tif
✅ Started export: NEXGDDP_CMIP6_Tekeze/INM-CM5-0/historical/tas/1981_11.tif
✅ Started export: NEXGDDP_CMIP6_Tekeze/INM-CM5-0/historical/tas/1981_12.tif
✅ Started expo